# Predict logP

The aim of this exercise is to compare model performance between a GNN and a supervised method to predict logP. 
Consider as a starting point one of the GNNs from session 14 (GCN, GIN, or GAT), and a supervised model of your choice (e.g., Random Forest with MACCS fingerprints).

#### Tasks:
1) Create a training and a test set
2) Build a supervised model of your choice on the training data and evaluate its performance on the test set
3) Build a GNN and compare its performance to the supervised model
4) Discuss the outcome


In [1]:
# complete imports if needed for your solution
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem import MACCSkeys
from rdkit.Chem import AllChem

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
import torch.optim as optim


ModuleNotFoundError: No module named 'torch_geometric'

Load the data from Session 11

In [4]:
df = pd.read_csv("esol_modified.csv").dropna(subset=["SMILES"])
df = df.loc[df.SMILES != 'C'] # remove the one compound containing only a single atom
df.head()

,SMILES,LogS,MolWt,LogP,EState_VSA5,TPSA,NumHAcc,NumAromaticRings,HeavyAtomCount,RingCount,qed,NumHDonors,NOCount
0,OCC3OC(OCC2OC(OC(C#N)c1ccccc1)C(O)C(O)C2O)C(O)...,-0.77,457.432,-3.10802,0.000000,202.32,12.0,1.0,32.0,3.0,0.217518,7.0,12.0
1,Cc1occc1C(=O)Nc2ccccc2,-3.30,201.225,2.84032,6.263163,42.24,2.0,2.0,15.0,2.0,0.811283,1.0,3.0
2,CC(C)=CCCC(C)=CC(=O),-2.06,152.237,2.87800,5.573105,17.07,1.0,0.0,11.0,0.0,0.343706,0.0,1.0
3,c1ccc2c(c1)ccc3c2ccc4c5ccccc5ccc43,-7.87,278.354,6.29940,43.089794,0.00,0.0,5.0,22.0,5.0,0.291526,0.0,0.0
4,c1ccsc1,-1.33,84.143,1.74810,0.000000,0.00,1.0,1.0,5.0,1.0,0.448927,0.0,0.0


### Setting the stage
Split the data into training and test sets. The test set will be used to compare model performance.


In [5]:
X=df["SMILES"].values
y=df["LogP"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Baseline supervised model
Choose a 
   - regression model (RF, SVR, kNN, Gradient Boost, ...) 
   - molecular descriptor (RDKit, Mordred, ...) or fingerprint (MACCS, Morgan, RDKit, ...)
      
Build a feature matrix and target vector. Add scaling if needed for your model.
Train the model on the training set and apply it to the test set.
Calculate performance metrics (R2, RMSE) for model performance on the test set.

In [7]:
def smiles_to_fp(smiles, radius=2, n_bits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(n_bits)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    return np.array(fp)

In [8]:

X_train_mols = np.array([smiles_to_fp(smiles) for smiles in X_train])
X_test_mols = np.array([smiles_to_fp(smiles) for smiles in X_test])

rf_model = RandomForestRegressor(n_estimators=200, random_state=42)
rf_model.fit(X_train_mols, y_train)

y_pred = rf_model.predict(X_test_mols)

mse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse:.4f}")
print(f"R^2 Score: {r2:.4f}")

[11:28:53] DEPRECATION WARNING: please use MorganGenerator
[11:28:53] DEPRECATION WARNING: please use MorganGenerator
[11:28:53] DEPRECATION WARNING: please use MorganGenerator
[11:28:53] DEPRECATION WARNING: please use MorganGenerator
[11:28:53] DEPRECATION WARNING: please use MorganGenerator
[11:28:53] DEPRECATION WARNING: please use MorganGenerator
[11:28:53] DEPRECATION WARNING: please use MorganGenerator
[11:28:53] DEPRECATION WARNING: please use MorganGenerator
[11:28:53] DEPRECATION WARNING: please use MorganGenerator
[11:28:53] DEPRECATION WARNING: please use MorganGenerator
[11:28:53] DEPRECATION WARNING: please use MorganGenerator
[11:28:53] DEPRECATION WARNING: please use MorganGenerator
[11:28:53] DEPRECATION WARNING: please use MorganGenerator
[11:28:53] DEPRECATION WARNING: please use MorganGenerator
[11:28:53] DEPRECATION WARNING: please use MorganGenerator
[11:28:53] DEPRECATION WARNING: please use MorganGenerator
[11:28:53] DEPRECATION WARNING: please use MorganGenerat

Mean Squared Error: 0.9195
R^2 Score: 0.7663


### Unpervised GNN model
Choose a GNN architecture
   - GCN, GIN, or GAT
      
Transform input smiles to graph objects using the atom and bond features. Build graphs for both training and test set. Train the GNN on the training set. Adapt architecture and parameters until you are happy with the performance. Apply the trained model on the test set (once!) and calculate model performance metrics

In [9]:
def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    x = []
    for atom in mol.GetAtoms():
        x.append(atom.GetAtomicNum())
    x = torch.tensor(x, dtype=torch.float).unsqueeze(1)
    
    edge_index = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        edge_index.append([i, j])
        edge_index.append([j, i])
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

    y_tensor = torch.tensor(y_train, dtype=torch.float) 
    
    return Data(x=x, edge_index=edge_index, y=y_tensor)

In [10]:
train_graphs = [smiles_to_graph(smiles) for smiles in X_train]
test_graphs = [smiles_to_graph(smiles) for smiles in X_test]

train_graphs = [g for g in train_graphs if g is not None]
test_graphs = [g for g in test_graphs if g is not None]

train_loader = DataLoader(train_graphs, batch_size=32, shuffle=True)
test_loader = DataLoader(test_graphs, batch_size=32)

NameError: name 'Data' is not defined

In [11]:
class GNN(torch.nn.Module):
    def __init__(self):
        super(GNN, self).__init__()
        self.conv1 = GCNConv(1, 64)
        self.conv2 = GCNConv(64, 128)
        self.fc1 = nn.Linear(128, 64)
        self.fc2 = nn.Linear(64, 1)
    
    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, data.batch)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

In [12]:
model = GNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

for epoch in range(20):
    model.train()
    total_loss = 0
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        output = model(data).squeeze()
        loss = loss_fn(output, data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

NameError: name 'GCNConv' is not defined

In [13]:
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for data in test_loader:
        data = data.to(device)
        output = model(data).squeeze()
        y_true.extend(data.y.cpu().numpy())
        y_pred.extend(output.cpu().numpy())

r2_gnn = r2_score(y_true, y_pred)
rmse_gnn = np.sqrt(mean_squared_error(y_true, y_pred))

print(f"GNN R^2 Score: {r2_gnn:.4f}")
print(f"GNN RMSE: {rmse_gnn:.4f}")

NameError: name 'model' is not defined

### Discussion points
1) Which model (supervised or unsupervised) performed better on the test set, and why?
RFR performed better, as GNN underperformed due to the limited data available

2) What did you try to improve model performance of your GNN? What did work, what did not work?
Inreased the depth or adding more layers worked good in order to prevent overfitting; further increasing the depth did not improve the model significantly and eventually resulted in overfitting

3) Which challenges did you face in the process of building the models?
Main challenge was converting the SMILES into graphs 

4) Which of your two models would you recommend to a chemist for predicting logP, and why?
If only a limited dataset is available RFR is easier, faster and more reliable; One advantage of GNN is that it can "visually" learn from 
uncommon/new structural motifs, where (standard) fingerprint system strugle